In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:

from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")


In [10]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [11]:
search_tool.invoke("What is the capital of France?")

[{'url': 'https://home.adelphi.edu/~ca19535/page%204.html',
  'content': '|  |  |  |  |  |  |  |\n ---  ---  --- \n| Home | Spain | Sydney | San Francisco | Paris | Las Vegas | Maui |\n\n  \n Paris, France\n\n  \n\n## Paris facts: Paris, the capital of France\n\nParis is the capital of France, the largest country of Europe with 550 000 km2 (65 millions inhabitants).\n\nParis has 2.234 million inhabitants end 2011. She is the core of Ile de France region (12 million people).\n\nFounded more than 2000 years ago, Paris is a modern and vibrant city with significant commercial, cultural, academic, scientific, administrative, business activity.\n\n## Paris facts: Paris history\n\nFrom the Romans to the Vth Republic, Paris has a rich 2000 year history. See details of Paris history. [...] ## Paris facts: the capital of France in history\n\nBefore Paris, the capital of France was Lyon (under the Romans). Paris first became the capital of France in 508 under King Clovis. After centuries with no 

In [12]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [13]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she had too many bugs in her system!', response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 13, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-3e56d360-0cb7-433c-ad4d-21782d44cc43-0')

In [14]:
prompt = hub.pull("hwchase17/react")

c:\Users\kumar.KUMAR\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [15]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [16]:
tools = [search_tool]

In [17]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [18]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


In [19]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
I should use the search engine to find the capital of India first and then search for its current weather.
Action: tavily_search_results_json
Action Input: "capital of India"[{'url': 'https://www.countryreports.org/country/India/capital-city.htm', 'content': '|  |  |\n --- |\n| Capital City | New Delhi |\n| Capital location | 28 36 N, 77 12 E |\n| Capital - history | the city\'s name is associated with various myths and legends; the original name for the city may have been Dhilli or Dhillika; alternatively, the name could be a corruption of the Hindustani words "dehleez" or "dehali" - both terms meaning "threshold" or "gateway" - and indicative of the city as a gateway to the Gangetic Plain; after the British decided to move the capital of their Indian Empire from Calcutta to Delhi in 1911, they created a new governmental district south of the latter designated as New Delhi; the new capital was not formally inaugurated until 1931. |\n| Capital - 

In [20]:
print(response["output"])

The capital of India is New Delhi, and the current weather there shows high temperatures between 94° to 113° with overnight lows between 78° and 89°.
